# Focus on version 1.0 as our assumption of this article's dataset
- https://www.tycho.pitt.edu/data/level3.php?utm_source=chatgpt.com, LEVEL 1 Data, version 1.0.0
- Article: https://academic.oup.com/ofid/article/5/7/ofy137/5039595

In [1]:
import pandas as pd

df = pd.read_csv('tycho/ProjectTycho_Level1_v1.0.0.csv', low_memory=False)
df.head()

,epi_week,state,loc,loc_type,disease,cases,incidence_per_100000
0,196601,MN,MINNESOTA,STATE,HEPATITIS A,3,0.08
1,196601,CO,COLORADO,STATE,HEPATITIS A,1,0.05
2,196601,AZ,ARIZONA,STATE,HEPATITIS A,6,0.37
3,196601,MT,MONTANA,STATE,HEPATITIS A,2,0.28
4,196601,LA,LOUISIANA,STATE,HEPATITIS A,1,0.03


In [2]:
df['year'] = (
    df['epi_week']
    .astype(str)
    .str[:4]
    .astype(int)
)

df['cases'] = pd.to_numeric(
    df['cases'],
    errors='coerce'
)

df['epi_week'] = pd.to_numeric(
    df['epi_week'],
    errors='coerce'
)

In [3]:
df.head()

,epi_week,state,loc,loc_type,disease,cases,incidence_per_100000,year
0,196601,MN,MINNESOTA,STATE,HEPATITIS A,3.0,0.08,1966
1,196601,CO,COLORADO,STATE,HEPATITIS A,1.0,0.05,1966
2,196601,AZ,ARIZONA,STATE,HEPATITIS A,6.0,0.37,1966
3,196601,MT,MONTANA,STATE,HEPATITIS A,2.0,0.28,1966
4,196601,LA,LOUISIANA,STATE,HEPATITIS A,1.0,0.03,1966


In [4]:
exclude_regions = [
    'HI',
    'AK',
]

measles_df = df[(df["disease"] == "MEASLES" )
                & (df['loc_type'] == "STATE")
                & (~df['state'].isin(exclude_regions))
                & (df['year'] >= 1931) 
                & (df['year'] <= 1992) 
                ].drop_duplicates()

measles_df['week'] = measles_df['epi_week'] % 100

measles_df.head()

,epi_week,state,loc,loc_type,disease,cases,incidence_per_100000,year,week
97355,193101,NY,NEW YORK,STATE,MEASLES,376.0,2.93,1931,1
97356,193101,OR,OREGON,STATE,MEASLES,67.0,6.94,1931,1
97357,193101,CO,COLORADO,STATE,MEASLES,41.0,3.88,1931,1
97358,193101,AZ,ARIZONA,STATE,MEASLES,50.0,11.66,1931,1
97359,193101,MO,MISSOURI,STATE,MEASLES,1160.0,31.26,1931,1


In [5]:
print("Regions:", measles_df['state'].nunique())
print("Rows:", len(measles_df))
print("First week:", measles_df['epi_week'].min())
print("Last week:", measles_df['epi_week'].max())
print("Total cases:", measles_df['cases'].sum())

Regions: 49
Rows: 128308
First week: 193101
Last week: 199252
Total cases: 17399916.0


# Let's start imputing - filling missing weeks
This is their most likely method as per chatgpt
- They probably imputed bounded internal gaps and left unbounded edge gaps unresolved/excluded when aggregating.

Essentailly, we ignore missing weeks in the boundaries (in sequence). Just fill the missing weeks in between min and max

In [6]:
essential_cols = ['week', 'state', 'cases', 'year']
measles_df = measles_df[essential_cols]

In [7]:
len(measles_df[(measles_df['week'] == 1) & (measles_df['year'] == 1931)]) # imputable boundaries

43

In [8]:
len(measles_df[(measles_df['week'] == 52) & (measles_df['year'] == 1992)]) # imputable boundaries

30

In [ ]:
missing_rows = pd.DataFrame(columns=essential_cols)
for state in measles_df['state'].unique():

    state_df = measles_df[measles_df['state'] == state].sort_values(by=['year', 'week'])

    for i in range(len(state_df)):
        current_row = state_df.iloc[i]

        # Next row
        if i + 1 < len(state_df):
            next_row = state_df.iloc[i + 1]
        else:
            next_row = None

        # we only start imputation if current row is not the first or the last
        if next_row is not None:

            # first we check that the next row is the following week from the current row
            current_week = current_row["week"]
            next_week = next_row["week"]

            week_diff = abs(next_week - current_week)

            # diffeence is greater than 1
            if week_diff > 1:

                if current_week > next_week: # this condition handles if week is transition to another year
                   missing_weeks = list(range(current_week + 1, 53))
                else:
                    # weeks are in same year
                    missing_weeks = list(range(current_week + 1, next_week))

                count_val = (current_row['cases'] + next_row['cases']) / 2
                for week in missing_weeks:

                    row_dict = {
                        'week': week,
                        'state':current_row['state'],
                        'cases': count_val,
                        'year': current_row['year']
                        }
                    
                    pd_series = pd.Series(row_dict)
                    missing_rows = pd.concat([missing_rows, pd_series.to_frame().T], ignore_index=True)

In [10]:
len(missing_rows) 

9165

In [11]:
len(missing_rows) / (len(missing_rows) + len(measles_df))

0.06666763655408699

In [12]:
assert False

AssertionError: 

In [ ]:
repro_measles_df = pd.concat([measles_df, missing_rows], ignore_index=True)

In [ ]:
prevaccination = repro_measles_df[(repro_measles_df['year'] >= 1931) & (repro_measles_df['year'] <= 1963)]
print(round(prevaccination['cases'].sum(), 2))

16650394.0


In [ ]:
introduction = repro_measles_df[(repro_measles_df['year'] >= 1964) & (repro_measles_df['year'] <= 1970)]
print(round(introduction['cases'].sum(), 2))

1076289.0


In [ ]:
one_dose_vaccine = repro_measles_df[(repro_measles_df['year'] >= 1971) & (repro_measles_df['year'] <= 1989)]
print(round(one_dose_vaccine['cases'].sum(), 2))

361624.5
